<a href="https://colab.research.google.com/github/J0SAL/genai-projects/blob/main/7_chroma_db/chroma_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -Uq chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

Initialize your client and create a collection. Feel free to give the collection a name you want, this is going to be the identifier of your collection so that you can retrieve it afterwards.

In [18]:
import chromadb
client = chromadb.Client()

collection = client.get_or_create_collection(name="my_collection")

In [25]:
collection.add(
    ids=["id5", "id6"],
    documents=[
        "This is a document about happy",
        "This is a document about sad"
    ]
)

In this example, we're adding a couple of documents to our collection. As you can see, we are passing the documents as simple text.

But of course, since vector databases need to embed the text before adding it, these documents are going to be processed with the sentence transformers locally within ChromaDB. All of this is done behind the scenes. You don't have to worry about it.

To get the documents from your collection, use the query method and provide the query text to find similar documents. You can also specify the number of results to return.

The text in your query text parameter is going to be embedded automatically by Chroma using the same embeddings model that was used to embed the documents that it ingested before: the sentence transformer model.

In [29]:
from pprint import pprint
results = collection.query(
    query_texts=["This is a query document about cry"], # Chroma will embed this for you
    n_results=2 # how many results to return
)
pprint(results)


{'data': None,
 'distances': [[0.8757300972938538, 1.0965479612350464]],
 'documents': [['This is a document about sad',
                'This is a document about joy']],
 'embeddings': None,
 'ids': [['id6', 'id3']],
 'included': ['metadatas', 'documents', 'distances'],
 'metadatas': [[None, None]],
 'uris': None}


Here, we can clearly see the results from Chroma. They are sorted by how close they are to our query. You can also view how far each document is from your query in the distances property.

## CRUD on Data Points

In [30]:
collection = client.get_or_create_collection(
    name="collection_j1",
    metadata={"description": "..."}
)

#### 1. Add

In [31]:
collection.add(
    ids=["1", "2", "3", "4", "5"],
    documents=[
      "The Eiffel Tower in Paris stands at 324 meters tall.",
      "Penguins can swim at speeds up to 22 miles per hour.",
      "The human body contains approximately 37.2 trillion cells.",
      "Mount Everest grows about 4 millimeters higher every year.",
      "The first email was sent in 1971 by Ray Tomlinson."
      ],
    metadatas=[
        {"source": "architecture", "location": "Paris", "year_built": 1889},
        {"source": "wildlife", "animal": "penguin", "habitat": "Antarctica"},
        {"source": "biology", "topic": "human anatomy", "fact_type": "cellular"},
        {"source": "geology", "mountain": "Everest", "fact_type": "growth"},
        {"source": "technology", "topic": "communication", "inventor": "Ray Tomlinson"}
    ],
)


#### 2. Update

If an id is not found in the collection, an error will be logged and the update will be ignored. If documents are supplied without corresponding embeddings, the embeddings will be recomputed with the collection's embedding function.

In [34]:
collection.update(
    ids=["5"],
    # embeddings = [[1.1, 2.3, 3.2], [2.1, 2.1, 2.4], ....]
    documents=[
        "This Colosseum is the largest ancient amphitheatre ever built",
    ],
    metadatas=[
        {"location": "Rome"}, # only updates the specified keys
    ]
)

There is also the possibility of using the upsert method, which updates a data point if it already exists; if it doesn't exist, it creates it

In [36]:
collection.upsert(
    ids=["1", "6"],
    # embeddings=[[1.1, 2.3, 3.2], [4.5, 6.9, 4.4], [1.1, 2.3, 3.2], ...],
    metadatas=[
      {"location": "Rome"},
      {"location": "Paris"},
    ],
    documents=[
      "The Colosseum is a Roman amphitheatre in the centre of the city of Rome, Italy.",
      "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France."],
)


#### 3. Get

In [37]:
collection.get(
    ids=["6"],
    include=["embeddings", "metadatas", "documents"] # default is ["metadatas", "documents"]
)


{'ids': ['6'],
 'embeddings': array([[ 4.45300005e-02,  6.78277314e-02,  5.09724719e-03,
          1.68784689e-02,  1.69589873e-02, -1.13394046e-02,
         -8.65743607e-02,  2.62315366e-02,  2.52917930e-02,
          1.02061451e-04, -1.78235043e-02, -3.55288647e-02,
          5.92213422e-02, -9.66211483e-02, -1.66444648e-02,
         -6.48750812e-02,  1.17878700e-02, -1.92353856e-02,
         -3.76679897e-02, -8.02256691e-04,  4.25267182e-02,
         -9.45624337e-02,  9.91592929e-03,  1.99287105e-02,
         -6.52991757e-02,  6.78890049e-02, -8.09928253e-02,
          1.00628994e-01, -1.09229898e-02, -6.67210817e-02,
          3.80184278e-02, -5.13828918e-02, -6.35576695e-02,
          7.24343359e-02, -4.61728238e-02,  3.88836265e-02,
          1.68099105e-02, -7.69589171e-02, -6.04594760e-02,
         -1.98722240e-02,  2.65931524e-03, -8.68613273e-03,
         -8.23829137e-03,  2.23441757e-02, -4.89933528e-02,
          3.53366765e-03,  1.69122368e-02, -1.78088062e-02,
         -1

#### Similarity Search

In [38]:
collection.query(
  query_texts=["Information about the capital of France"],
  n_results=2
)

{'ids': [['6', '1']],
 'embeddings': None,
 'documents': [['The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.',
   'The Colosseum is a Roman amphitheatre in the centre of the city of Rome, Italy.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'location': 'Paris'},
   {'year_built': 1889, 'location': 'Rome', 'source': 'architecture'}]],
 'distances': [[1.259374976158142, 1.6846143007278442]]}

## CRUD on Collections

You can use the `get_or_create_collection` method to get a collection by name. And if it doesn't exist, it will be created and the method will return that newly created collection.

Also notice that we can add custom metadata, which are arbitrary key-value pairs with information about your collection.


In [46]:
collection = client.get_or_create_collection(
    name="my_collection",
    metadata={"description": "..."}
)


Get Collections

In [40]:
collections = client.list_collections()
print(collections)

[Collection(name=my_collection), Collection(name=collection_j1)]


By default, list_collections returns up to 100 collections. If you want to go through the entire list of collections, you're going to have to do something like this:

In [41]:
batch_size = 100
offset = 0
all_collections = []

while True:
    collections_batch = client.list_collections(limit=batch_size, offset=offset)
    if not collections_batch:  # If no more collections are returned
        break
    all_collections.extend(collections_batch)
    offset += batch_size

print(all_collections)

[Collection(name=my_collection), Collection(name=collection_j1)]


### Patch collection
You can update a collection's information using the modify method.

In [42]:
collection.modify(
   name="my_newer_collection",
   metadata={"description": "this is a great collection of data points"}
)


In [43]:
print(client.list_collections())

[Collection(name=my_newer_collection), Collection(name=collection_j1)]


In [44]:
client.delete_collection(name="my_newer_collection")
print(client.list_collections())


[Collection(name=collection_j1)]


### Convenience methods

In [47]:
collection.peek(
  limit=2
)

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

## Persistent Database

In [48]:
import chromadb

client = chromadb.PersistentClient(path="./chroma")
collection = client.get_or_create_collection(name="rag_documents")

In [50]:
# remote instance - need the instance up and running
chroma_client = chromadb.HttpClient(host='localhost', port=8000)

# managed cloud instance
client = chromadb.CloudClient(
    tenant='Tenant ID',
    database='Database name',
    api_key='Chroma Cloud API key'
)

# OR
# If you set the CHROMA_API_KEY, CHROMA_TENANT, and the CHROMA_DATABASE environment variables, you can simply instantiate a CloudClient with no arguments:

client = chromadb.CloudClient()

ValueError: Could not connect to a Chroma server. Are you sure it is running?